# MQ-009: Medical RAG Retrieval Evaluation

## Research Objective

Evaluate semantic retrieval quality using the
MedQuAD medical knowledge corpus and FAISS.

## Previously Completed

- Cleaned medical records: 15,864
- Final medical passages: 38,544
- Embedding model: all-MiniLM-L6-v2
- Embedding dimensions: 384
- Vector index: FAISS IndexFlatIP
- Similarity: Cosine-equivalent inner product

## Notebook Scope

This notebook loads previously generated artifacts.

It does not repeat:
- Medical data preprocessing
- Passage chunking
- Embedding generation
- FAISS index construction

## Evaluation Plan

1. Load saved retrieval artifacts.
2. Load held-out validation questions.
3. Perform semantic retrieval.
4. Evaluate retrieval relevance.
5. Record experimental findings.

In [1]:
from pathlib import Path

import faiss
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    "/home/sagemaker-user/medical_chatbot"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load passage metadata
passages_df = pd.read_csv(
    PROCESSED_DIR / "medquad_final_passages.csv"
)

# Load existing FAISS index
faiss_index = faiss.read_index(
    str(PROCESSED_DIR / "medquad_faiss.index")
)

# Validate artifact consistency
assert len(passages_df) == faiss_index.ntotal
assert faiss_index.d == 384

print("Medical passages:", len(passages_df))
print("FAISS indexed vectors:", faiss_index.ntotal)
print("Embedding dimensions:", faiss_index.d)
print("Artifacts loaded successfully!")

Medical passages: 38544
FAISS indexed vectors: 38544
Embedding dimensions: 384
Artifacts loaded successfully!


In [2]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print(
    "Embedding dimensions:",
    embedding_model.get_embedding_dimension()
)

print("Query embedding model ready!")

/home/sagemaker-user/medquad-envs/rag-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1432.00it/s]


Embedding dimensions: 384
Query embedding model ready!


In [3]:
def retrieve_medical_passages(query, top_k=5):

    # Encode only the new query
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cpu"
    ).astype(np.float32)

    # Search existing FAISS index
    scores, indices = faiss_index.search(
        query_vector,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx < 0:
            continue

        record = passages_df.iloc[int(idx)]

        results.append({
            "rank": rank,
            "similarity": float(score),
            "question": record["question"],
            "passage": record["passage"],
            "source": record["source"],
            "focus_area": record["focus_area"]
        })

    return pd.DataFrame(results)

In [4]:
query = "What are the symptoms of chronic kidney disease?"

results = retrieve_medical_passages(
    query,
    top_k=5
)

display(
    results[
        [
            "rank",
            "similarity",
            "question",
            "source",
            "focus_area"
        ]
    ]
)

,rank,similarity,question,source,focus_area
0,1,0.715603,What are the symptoms of Kidney Disease ?,NIHSeniorHealth,Kidney Disease
1,2,0.686266,What is (are) Chronic Kidney Disease ?,MPlusHealthTopics,Chronic Kidney Disease
2,3,0.677963,What are the symptoms of Atherosclerosis ?,NHLBI,Atherosclerosis
3,4,0.676642,What are the symptoms of Kidney Disease ?,NIHSeniorHealth,Kidney Disease
4,5,0.665614,What are the complications of Polycystic Kidne...,NIDDK,Polycystic Kidney Disease


## MQ-009: Retrieved Evidence Inspection

### Objective
Assess whether retrieved medical passages directly
answer the user's question.

### Evaluation Criteria
- Medical condition relevance
- Question-answer relevance
- Presence of useful clinical information
- Potentially misleading or unrelated evidence

### Research Importance
High embedding similarity does not necessarily imply
high clinical relevance.

The inspection will help identify whether additional
retrieval filtering or reranking is needed.

In [5]:
query = "What are the symptoms of chronic kidney disease?"

results = retrieve_medical_passages(query, top_k=5)

for row in results.itertuples(index=False):

    print(f"\nRank: {row.rank}")
    print(f"Similarity: {row.similarity:.4f}")
    print(f"Source: {row.source}")
    print(f"Focus area: {row.focus_area}")
    print(f"Original question: {row.question}")

    print("\nMedical evidence:")
    print(row.passage)

    print("\n" + "=" * 80)


Rank: 1
Similarity: 0.7156
Source: NIHSeniorHealth
Focus area: Kidney Disease
Original question: What are the symptoms of Kidney Disease ?

Medical evidence:
appetite, and some have leg cramps. These problems are caused by waste products that build up in the blood, a condition known as uremia. Healthy kidneys remove waste products from the blood. When the kidneys stop working, uremia occurs. The kidneys also make hormones and balance the minerals in the blood. When the kidneys stop working, most people develop conditions that affect the blood, bones, nerves, and skin. These problems may include itching, sleep problems, restless legs, weak bones, joint problems, and depression. How Kidney Disease Is Diagnosed Blood and urine tests are the only way to check for kidney damage or measure kidney function. It is important for you to get checked for kidney disease if you have the


Rank: 2
Similarity: 0.6863
Source: MPlusHealthTopics
Focus area: Chronic Kidney Disease
Original question: What

In [6]:
from pathlib import Path

# Retrieve baseline results
query = "What are the symptoms of chronic kidney disease?"

results = retrieve_medical_passages(
    query,
    top_k=5
)

# Create evaluation output directory
output_dir = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "data/processed/retrieval_evaluation"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Save full passages without truncation
output_path = output_dir / "ckd_baseline_retrieval.csv"

results.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)
print("Retrieved passages:", len(results))

# Display one passage at a time to avoid output truncation
for _, row in results.iterrows():
    print(
        f"Rank {row['rank']} | "
        f"Similarity {row['similarity']:.4f} | "
        f"{row['focus_area']}"
    )
    print("Passage length:", len(row["passage"]))

Saved: /home/sagemaker-user/medical_chatbot/data/processed/retrieval_evaluation/ckd_baseline_retrieval.csv
Retrieved passages: 5
Rank 1 | Similarity 0.7156 | Kidney Disease
Passage length: 727
Rank 2 | Similarity 0.6863 | Chronic Kidney Disease
Passage length: 689
Rank 3 | Similarity 0.6780 | Atherosclerosis
Passage length: 754
Rank 4 | Similarity 0.6766 | Kidney Disease
Passage length: 719
Rank 5 | Similarity 0.6656 | Polycystic Kidney Disease
Passage length: 677


## MQ-010: Held-Out Retrieval Evaluation

### Objective
Evaluate the baseline FAISS retriever using 50 held-out
medical questions.

### Dataset
- Validation dataset: validation.csv
- Storage: Amazon S3
- Retrieval corpus: 38,544 indexed passages

### Evaluation Strategy
The validation questions will be encoded using the
existing sentence-transformer model.

Retrieved passages will be assessed for medical topic
relevance and answer usefulness.

### Research Limitations
A matching medical focus area does not guarantee that
a passage correctly answers the question.

Human relevance judgments or independently verified
reference evidence will be needed for stronger
retrieval-quality conclusions.

In [7]:
import boto3
import pandas as pd
from io import BytesIO

s3 = boto3.client("s3")

response = s3.get_object(
    Bucket="medquad-ai-cleaned",
    Key="validation.csv"
)

validation_df = pd.read_csv(
    BytesIO(response["Body"].read())
)

print("Validation records:", len(validation_df))
print("Columns:", validation_df.columns.tolist())

print("\nMissing values:")
print(validation_df.isnull().sum())

print("\nSample validation questions:")
display(
    validation_df[
        ["question", "focus_area"]
    ].head(10)
)

Validation records: 50
Columns: ['question', 'answer', 'source', 'focus_area', 'input_text', 'target_text', 'input_token_length', 'target_token_length']

Missing values:
question               0
answer                 0
source                 0
focus_area             0
input_text             0
target_text            0
input_token_length     0
target_token_length    0
dtype: int64

Sample validation questions:


,question,focus_area
0,Is Galactosialidosis inherited ?,Galactosialidosis
1,What are the symptoms of Coronary Microvascula...,Coronary Microvascular Disease
2,What is (are) Schwartz-Jampel syndrome ?,Schwartz-Jampel syndrome
3,What is (are) Menetrier disease ?,Menetrier disease
4,What causes Hanhart syndrome ?,Hanhart syndrome
5,What is (are) Breast Cancer ?,Breast Cancer
6,What are the treatments for MTHFR gene mutation ?,MTHFR gene mutation
7,What are the treatments for aromatic l-amino a...,aromatic l-amino acid decarboxylase deficiency
8,What is (are) Sinusitis ?,Sinusitis
9,What are the treatments for Crimean-Congo Hemo...,Crimean-Congo Hemorrhagic Fever (CCHF)


## MQ-010: Baseline Retrieval Topic-Match Evaluation

### Objective
Measure whether FAISS retrieves passages associated
with the same medical focus area as held-out questions.

### Evaluation Dataset
50 validation questions from MedQuAD.

### Metrics
- Focus Area Match@1
- Focus Area Match@3
- Focus Area Match@5

A match occurs when at least one retrieved passage
has the same normalized focus-area label as the query.

### Limitations
Focus-area matching is an imperfect relevance proxy.
Related conditions may have different labels, while
same-topic passages may not answer the question.

These metrics must not be interpreted as clinical
factuality, answer correctness, or true Recall@K.

In [8]:
import numpy as np
import pandas as pd

# Normalize focus-area labels
def normalize_focus_area(value):
    return " ".join(str(value).casefold().split())


# Encode only the 50 validation questions
validation_vectors = embedding_model.encode(
    validation_df["question"].tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device="cpu",
    show_progress_bar=True
).astype(np.float32)

# Search existing FAISS index
scores, indices = faiss_index.search(
    validation_vectors,
    k=5
)

evaluation_records = []

for i, row in validation_df.iterrows():

    expected_focus = normalize_focus_area(
        row["focus_area"]
    )

    retrieved_focus_areas = [
        normalize_focus_area(
            passages_df.iloc[int(idx)]["focus_area"]
        )
        for idx in indices[i]
        if idx >= 0
    ]

    matches = [
        focus == expected_focus
        for focus in retrieved_focus_areas
    ]

    evaluation_records.append({
        "question": row["question"],
        "expected_focus_area": row["focus_area"],
        "match_at_1": any(matches[:1]),
        "match_at_3": any(matches[:3]),
        "match_at_5": any(matches[:5]),
        "top_1_similarity": float(scores[i][0]),
        "retrieved_focus_areas": " | ".join(
            retrieved_focus_areas
        )
    })

evaluation_df = pd.DataFrame(evaluation_records)

print("Evaluated questions:", len(evaluation_df))

print(
    "Focus Area Match@1:",
    f"{evaluation_df['match_at_1'].mean():.2%}"
)

print(
    "Focus Area Match@3:",
    f"{evaluation_df['match_at_3'].mean():.2%}"
)

print(
    "Focus Area Match@5:",
    f"{evaluation_df['match_at_5'].mean():.2%}"
)

print("\nQuestions without a matching focus area in Top 5:")

display(
    evaluation_df.loc[
        ~evaluation_df["match_at_5"],
        [
            "question",
            "expected_focus_area",
            "retrieved_focus_areas"
        ]
    ].head(10)
)

Batches: 100%|██████████| 2/2 [00:00<00:00,  4.33it/s]


Evaluated questions: 50
Focus Area Match@1: 66.00%
Focus Area Match@3: 72.00%
Focus Area Match@5: 82.00%

Questions without a matching focus area in Top 5:


,question,expected_focus_area,retrieved_focus_areas
5,What is (are) Breast Cancer ?,Breast Cancer,male breast cancer | male breast cancer | male...
8,What is (are) Sinusitis ?,Sinusitis,preauricular sinus | preauricular sinus | sick...
11,What are the genetic changes related to mucopo...,mucopolysaccharidosis type II,mucopolysaccharidosis type iv | mucopolysaccha...
12,What is (are) Dendritic cell tumor ?,Dendritic cell tumor,desmoplastic small round cell tumor | paraneop...
13,What is (are) Pinworms ?,Pinworms,parasites - enterobiasis (also known as pinwor...
21,Do you have information about Child Care,Child Care,premature babies | children's health | prenata...
42,What are the symptoms of Madelung disease ?,Madelung disease,whipple disease | churg strauss syndrome | bal...
43,Do you have information about Prostate Cancer ...,Prostate Cancer Screening,prostate cancer | prostate cancer | prostate c...
47,What are the symptoms of Cobb syndrome ?,Cobb syndrome,beare-stevenson cutis gyrata syndrome | townes...


## MQ-011: Retrieval Failure Diagnosis

### Objective
Determine whether unmatched validation questions
represent missing knowledge coverage or retrieval
ranking problems.

### Methodology
1. Identify validation questions without a matching
   focus area in the top five results.
2. Search the complete retrieval metadata for an
   exact normalized focus-area match.
3. Count available passages for each medical topic.

### Interpretation
- Zero matching passages: Potential coverage or
  terminology mismatch.
- One or more matching passages: The topic is
  represented, but FAISS did not retrieve its exact
  focus-area label in the top five.

### Limitation
Exact focus-area matching cannot identify medical
synonyms or verify passage-level clinical relevance.

In [9]:
# MQ-011: Diagnose unmatched validation questions

corpus_focus = (
    passages_df["focus_area"]
    .fillna("")
    .map(normalize_focus_area)
)

unmatched_df = evaluation_df.loc[
    ~evaluation_df["match_at_5"]
].copy()

diagnostic_records = []

for row in unmatched_df.itertuples(index=False):

    expected = normalize_focus_area(
        row.expected_focus_area
    )

    available_passages = int(
        (corpus_focus == expected).sum()
    )

    diagnostic_records.append({
        "question": row.question,
        "expected_focus_area": row.expected_focus_area,
        "available_passages": available_passages,
        "exact_focus_area_in_corpus": (
            available_passages > 0
        ),
        "diagnosis": (
            "Potential ranking failure"
            if available_passages > 0
            else "Coverage or terminology mismatch"
        )
    })

diagnostic_df = pd.DataFrame(diagnostic_records)

display(diagnostic_df)

print("\nTotal unmatched:", len(diagnostic_df))

print(
    "Topics with exact focus-area coverage:",
    diagnostic_df["exact_focus_area_in_corpus"].sum()
)

print(
    "Topics without exact focus-area coverage:",
    (~diagnostic_df["exact_focus_area_in_corpus"]).sum()
)

,question,expected_focus_area,available_passages,exact_focus_area_in_corpus,diagnosis
0,What is (are) Breast Cancer ?,Breast Cancer,273,True,Potential ranking failure
1,What is (are) Sinusitis ?,Sinusitis,0,False,Coverage or terminology mismatch
2,What are the genetic changes related to mucopo...,mucopolysaccharidosis type II,8,True,Potential ranking failure
3,What is (are) Dendritic cell tumor ?,Dendritic cell tumor,0,False,Coverage or terminology mismatch
4,What is (are) Pinworms ?,Pinworms,0,False,Coverage or terminology mismatch
5,Do you have information about Child Care,Child Care,0,False,Coverage or terminology mismatch
6,What are the symptoms of Madelung disease ?,Madelung disease,8,True,Potential ranking failure
7,Do you have information about Prostate Cancer ...,Prostate Cancer Screening,0,False,Coverage or terminology mismatch
8,What are the symptoms of Cobb syndrome ?,Cobb syndrome,0,False,Coverage or terminology mismatch



Total unmatched: 9
Topics with exact focus-area coverage: 3
Topics without exact focus-area coverage: 6


## MQ-012: Retrieval Ranking Depth Analysis

### Objective
Determine whether medical passages associated with
the correct focus area appear beyond the top five
retrieval results.

### Methodology
1. Select questions with exact focus-area coverage.
2. Retrieve the top 50 passages using the existing
   FAISS index.
3. Identify the rank of the first matching focus area.
4. Record the similarity score at that rank.

### Research Significance
If relevant topic labels appear at deeper ranks,
reranking a larger candidate pool may improve
top-five retrieval performance.

### Limitations
Matching a focus area does not guarantee that a
passage answers the question or is clinically correct.

In [10]:
# MQ-012: Analyze retrieval depth

ranking_candidates = diagnostic_df[
    diagnostic_df["exact_focus_area_in_corpus"]
].copy()

ranking_results = []

for row in ranking_candidates.itertuples(index=False):

    query_vector = embedding_model.encode(
        [row.question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cpu"
    ).astype(np.float32)

    scores, indices = faiss_index.search(
        query_vector,
        k=50
    )

    expected_focus = normalize_focus_area(
        row.expected_focus_area
    )

    first_match_rank = None
    first_match_score = None

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx < 0:
            continue

        retrieved_focus = normalize_focus_area(
            passages_df.iloc[int(idx)]["focus_area"]
        )

        if retrieved_focus == expected_focus:
            first_match_rank = rank
            first_match_score = float(score)
            break

    ranking_results.append({
        "question": row.question,
        "focus_area": row.expected_focus_area,
        "available_passages": row.available_passages,
        "first_match_rank": first_match_rank,
        "first_match_similarity": first_match_score
    })

ranking_depth_df = pd.DataFrame(ranking_results)

display(ranking_depth_df)

,question,focus_area,available_passages,first_match_rank,first_match_similarity
0,What is (are) Breast Cancer ?,Breast Cancer,273,8,0.644317
1,What are the genetic changes related to mucopo...,mucopolysaccharidosis type II,8,9,0.707559
2,What are the symptoms of Madelung disease ?,Madelung disease,8,6,0.573008


## MQ-013: Cross-Encoder Reranking Experiment

### Research Question
Can cross-encoder reranking improve the relevance
of retrieved medical evidence compared with
FAISS cosine-similarity ranking alone?

### Baseline
- Embedding model: all-MiniLM-L6-v2
- Vector index: FAISS IndexFlatIP
- Focus Area Match@5: 82%

### Experimental Method
1. Retrieve 20 candidate passages using FAISS.
2. Score each query-passage pair with a cross-encoder.
3. Rerank candidates by cross-encoder score.
4. Select the top five passages.
5. Compare results against the existing FAISS baseline.

### Model
cross-encoder/ms-marco-MiniLM-L-6-v2

### Limitations
The reranker is trained primarily for general
information retrieval, not clinical reasoning.

Higher topic-match scores do not necessarily imply
greater medical factuality or safety.

In [11]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = (
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    device="cpu"
)

print("Reranker loaded successfully!")
print("Model:", RERANKER_MODEL_NAME)
print("Device: CPU")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1218.71it/s]


Reranker loaded successfully!
Model: cross-encoder/ms-marco-MiniLM-L-6-v2
Device: CPU


## MQ-016: Two-Stage Medical Evidence Retrieval

### Objective
Improve medical evidence ranking by applying a cross-encoder
to passages retrieved from the existing FAISS index.

### Stage 1: Semantic Candidate Retrieval
- Embedding model: all-MiniLM-L6-v2
- Vector index: FAISS IndexFlatIP
- Candidate passages: Top 20

### Stage 2: Cross-Encoder Reranking
- Model: cross-encoder/ms-marco-MiniLM-L-6-v2
- Input: User question and candidate medical evidence
- Output: Top 5 reranked passages

### Research Hypothesis
Cross-encoder reranking may improve question-specific
medical evidence relevance compared with FAISS alone.

### Experimental Constraints
- Preserve the original FAISS index.
- Do not regenerate passage embeddings.
- Preserve source metadata and original FAISS scores.
- Do not use validation labels to influence ranking.

### Limitations
The reranker is not medically specialized.
Higher ranking scores do not establish clinical correctness.

In [12]:
import numpy as np
import pandas as pd

def retrieve_and_rerank(query, candidate_k=20, top_k=5):

    if not 1 <= top_k <= candidate_k:
        raise ValueError("Require 1 <= top_k <= candidate_k")

    # Stage 1: Encode the medical question
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cpu"
    ).astype(np.float32)

    # Stage 2: Retrieve FAISS candidates
    scores, indices = faiss_index.search(
        query_vector,
        candidate_k
    )

    candidates = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx < 0:
            continue

        record = passages_df.iloc[int(idx)]

        candidates.append({
            "passage_id": int(idx),
            "faiss_rank": rank,
            "faiss_score": float(score),
            "question": record["question"],
            "passage": record["passage"],
            "source": record["source"],
            "focus_area": record["focus_area"]
        })

    if not candidates:
        return pd.DataFrame()

    # Stage 3: Prepare query-evidence pairs
    pairs = [
        (
            query,
            f"Question: {item['question']}\n"
            f"Passage: {item['passage']}"
        )
        for item in candidates
    ]

    # Stage 4: Cross-encoder inference
    reranker_scores = reranker.predict(
        pairs,
        batch_size=16,
        show_progress_bar=False
    )

    for item, score in zip(candidates, reranker_scores):
        item["reranker_score"] = float(score)

    # Stage 5: Rank candidates
    reranked_df = pd.DataFrame(candidates)

    reranked_df = (
        reranked_df
        .sort_values(
            ["reranker_score", "faiss_rank"],
            ascending=[False, True]
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    reranked_df.insert(
        0,
        "reranked_rank",
        range(1, len(reranked_df) + 1)
    )

    return reranked_df

In [13]:
# Reuse the exact validation questions from MQ-012
test_cases = ranking_candidates[
    ["question", "expected_focus_area"]
].copy()

for row in test_cases.itertuples(index=False):

    print("\n" + "=" * 75)
    print("QUERY:", row.question)
    print("EXPECTED FOCUS:", row.expected_focus_area)

    results = retrieve_and_rerank(
        query=row.question,
        candidate_k=20,
        top_k=5
    )

    results["focus_match"] = (
        results["focus_area"].map(normalize_focus_area)
        == normalize_focus_area(row.expected_focus_area)
    )

    display(
        results[
            [
                "reranked_rank",
                "faiss_rank",
                "faiss_score",
                "reranker_score",
                "focus_area",
                "focus_match"
            ]
        ]
    )

    print(
        "Exact focus-area match in Top 5:",
        results["focus_match"].any()
    )


QUERY: What is (are) Breast Cancer ?
EXPECTED FOCUS: Breast Cancer


,reranked_rank,faiss_rank,faiss_score,reranker_score,focus_area,focus_match
0,1,1,0.678464,5.263126,Male Breast Cancer,False
1,2,2,0.674003,4.802766,Male Breast Cancer,False
2,3,5,0.662312,4.595370,Metaplastic carcinoma of the breast,False
3,4,9,0.634207,4.404886,Male Breast Cancer,False
4,5,7,0.647544,4.359292,Cancer,False


Exact focus-area match in Top 5: False

QUERY: What are the genetic changes related to mucopolysaccharidosis type II ?
EXPECTED FOCUS: mucopolysaccharidosis type II


,reranked_rank,faiss_rank,faiss_score,reranker_score,focus_area,focus_match
0,1,6,0.721728,6.770056,mucopolysaccharidosis type III,False
1,2,2,0.786069,6.697238,mucopolysaccharidosis type I,False
2,3,1,0.816401,6.599124,mucopolysaccharidosis type IV,False
3,4,3,0.783205,6.537385,mucopolysaccharidosis type VII,False
4,5,7,0.711761,6.481394,mucopolysaccharidosis type IV,False


Exact focus-area match in Top 5: False

QUERY: What are the symptoms of Madelung disease ?
EXPECTED FOCUS: Madelung disease


,reranked_rank,faiss_rank,faiss_score,reranker_score,focus_area,focus_match
0,1,17,0.550849,7.570786,Madelung disease,True
1,2,11,0.558231,5.936464,Madelung disease,True
2,3,6,0.573008,2.853610,Madelung disease,True
3,4,1,0.589163,-0.928492,Whipple Disease,False
4,5,8,0.562165,-1.195129,Hashimoto's Disease,False


Exact focus-area match in Top 5: True


### MQ-016: Preliminary Reranking Findings

The FAISS + Cross-Encoder retrieval pipeline was
successfully implemented and tested on three
previously unmatched validation questions.

#### Observations
- Breast Cancer: No exact focus-area improvement.
- Mucopolysaccharidosis type II: No improvement.
- Madelung disease: Improved from FAISS rank 6
  to reranked rank 1.

#### Research Finding
Cross-encoder reranking can improve the ordering
of some medical evidence but may fail to distinguish
closely related diseases and subtypes.

#### Limitations
These observations are based on three selected
questions. They do not establish overall improvement.

Exact focus-area agreement is a proxy metric,
not a measure of clinical factuality.

In [14]:
# MQ-017: Evaluate FAISS + Cross-Encoder on validation data

reranking_records = []

for i, row in validation_df.iterrows():

    results = retrieve_and_rerank(
        query=row["question"],
        candidate_k=20,
        top_k=5
    )

    expected_focus = normalize_focus_area(
        row["focus_area"]
    )

    matches = (
        results["focus_area"]
        .map(normalize_focus_area)
        .eq(expected_focus)
        .tolist()
    )

    reranking_records.append({
        "question": row["question"],
        "expected_focus_area": row["focus_area"],
        "match_at_1": any(matches[:1]),
        "match_at_3": any(matches[:3]),
        "match_at_5": any(matches[:5]),
        "retrieved_focus_areas": " | ".join(
            results["focus_area"].tolist()
        )
    })

    if (i + 1) % 10 == 0:
        print(f"Evaluated {i + 1}/50 questions")

reranking_eval_df = pd.DataFrame(reranking_records)

print("\nFAISS + Cross-Encoder Results")

for k in [1, 3, 5]:
    score = reranking_eval_df[f"match_at_{k}"].mean()
    print(f"Focus Area Match@{k}: {score:.2%}")

Evaluated 10/50 questions
Evaluated 20/50 questions
Evaluated 30/50 questions
Evaluated 40/50 questions
Evaluated 50/50 questions

FAISS + Cross-Encoder Results
Focus Area Match@1: 54.00%
Focus Area Match@3: 70.00%
Focus Area Match@5: 74.00%


## MQ-017: Baseline vs Cross-Encoder Evaluation

### Experimental Setup
- Validation questions: 50
- Baseline: FAISS IndexFlatIP, Top 5
- Experimental system: FAISS Top 20 + Cross-Encoder Top 5
- Embedding model: all-MiniLM-L6-v2
- Reranker: ms-marco-MiniLM-L-6-v2

### Results

| Metric | FAISS | FAISS + Reranker |
|---|---:|---:|
| Focus Area Match@1 | 66% | 54% |
| Focus Area Match@3 | 72% | 70% |
| Focus Area Match@5 | 82% | 74% |

### Findings
Cross-encoder reranking reduced exact focus-area
matching across all evaluated retrieval depths.

The largest reduction occurred at Match@1,
with a decrease of 12 percentage points.

### Research Decision
Retain FAISS as the current baseline.

Do not adopt the cross-encoder as the default
retrieval strategy until further evaluation.

### Limitations
Focus-area matching is a proxy for topic relevance.
The experiment does not measure clinical factuality
or answer correctness.

In [15]:
from pathlib import Path
import pandas as pd

output_dir = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "data/processed/retrieval_evaluation"
)

output_dir.mkdir(parents=True, exist_ok=True)

comparison_df = evaluation_df[
    ["question", "expected_focus_area",
     "match_at_1", "match_at_3", "match_at_5"]
].merge(
    reranking_eval_df[
        ["question", "match_at_1", "match_at_3", "match_at_5"]
    ],
    on="question",
    suffixes=("_faiss", "_reranker"),
    validate="one_to_one"
)

comparison_df.to_csv(
    output_dir / "faiss_vs_reranker_validation.csv",
    index=False
)

print("Comparison saved successfully!")
print("Compared questions:", len(comparison_df))

Comparison saved successfully!
Compared questions: 50


## MQ-018: Manual Medical Evidence Relevance Assessment

### Objective
Evaluate the clinical-topic relevance and question-specific
usefulness of medical passages retrieved by FAISS and the
cross-encoder reranker.

### Evaluation Method
A human reviewer will assess retrieved passages using a
four-level relevance scale.

| Grade | Classification | Definition |
|---|---|---|
| 3 | Highly relevant | Correct medical condition and directly answers the question |
| 2 | Partially relevant | Correct condition but incomplete or indirect answer |
| 1 | Weakly relevant | Related medical topic but does not adequately answer the question |
| 0 | Irrelevant | Unrelated, misleading, or wrong medical condition |

### Error Categories
- Disease or subtype confusion
- Incorrect question intent (symptoms, causes, treatment, etc.)
- Incomplete evidence
- Related but non-specific medical evidence
- Missing corpus coverage
- Retrieval ranking failure

### Experimental Design
1. Select validation questions for manual review.
2. Retrieve Top 5 passages from FAISS.
3. Retrieve Top 5 passages after reranking.
4. Assign relevance grades to each passage.
5. Record the reason for each low relevance grade.
6. Compare retrieval relevance across both methods.

### Research Limitations
Relevance does not establish medical factual accuracy.
Human annotations are subjective and should be reviewed
consistently using the same rubric.

No retrieved passage will be used as personalized
medical advice.

In [16]:
import pandas as pd
from pathlib import Path

# Compare FAISS and reranker results
review_comparison = evaluation_df[
    ["question", "expected_focus_area", "match_at_5"]
].merge(
    reranking_eval_df[
        ["question", "match_at_5"]
    ],
    on="question",
    suffixes=("_faiss", "_reranker"),
    validate="one_to_one"
)

# Classify changes in retrieval outcome
def classify_outcome(row):
    if row["match_at_5_faiss"] and not row["match_at_5_reranker"]:
        return "Regression"

    if not row["match_at_5_faiss"] and row["match_at_5_reranker"]:
        return "Improvement"

    return "Unchanged"

review_comparison["outcome"] = review_comparison.apply(
    classify_outcome,
    axis=1
)

# Select up to 5 regressions and 5 non-regressions
regressions = review_comparison[
    review_comparison["outcome"] == "Regression"
].head(5)

other_cases = review_comparison[
    review_comparison["outcome"] != "Regression"
].head(10 - len(regressions))

selected_questions = pd.concat(
    [regressions, other_cases],
    ignore_index=True
)

print("Selected validation questions:", len(selected_questions))

display(
    selected_questions[
        [
            "question",
            "expected_focus_area",
            "outcome"
        ]
    ]
)

Selected validation questions: 10


,question,expected_focus_area,outcome
0,what research (or clinical trials) is being do...,Klver-Bucy Syndrome,Regression
1,What are the treatments for 3MC syndrome ?,3MC syndrome,Regression
2,what research (or clinical trials) is being do...,Isaacs' Syndrome,Regression
3,what research (or clinical trials) is being do...,Lung Cancer,Regression
4,What are the genetic changes related to Wiskot...,Wiskott-Aldrich syndrome,Regression
5,Is Galactosialidosis inherited ?,Galactosialidosis,Unchanged
6,What are the symptoms of Coronary Microvascula...,Coronary Microvascular Disease,Unchanged
7,What is (are) Schwartz-Jampel syndrome ?,Schwartz-Jampel syndrome,Unchanged
8,What is (are) Menetrier disease ?,Menetrier disease,Unchanged
9,What causes Hanhart syndrome ?,Hanhart syndrome,Unchanged


### MQ-018: Medical Evidence Review Dataset

**Objective:** Construct a reproducible human evaluation dataset
to compare medical evidence retrieved by FAISS and cross-encoder
reranking.

**Evaluation sample:**
- 10 validation questions
- 5 FAISS results per question
- 5 cross-encoder results per question
- 100 retrieval result rows before deduplication

**Reviewer annotations:**
- Relevance grade (0–3)
- Error category
- Reviewer comments

**Important:** Human annotations must be assigned by a reviewer
after reading the retrieved passage. They must not be inferred
automatically from the focus-area label.

In [17]:
review_records = []

for question_id, row in selected_questions.iterrows():

    query = row["question"]
    expected_focus = row["expected_focus_area"]

    # Original FAISS retrieval
    faiss_results = retrieve_medical_passages(
        query=query,
        top_k=5
    )

    # Cross-encoder reranking
    reranked_results = retrieve_and_rerank(
        query=query,
        candidate_k=20,
        top_k=5
    )

    # Record FAISS results
    for _, result in faiss_results.iterrows():

        review_records.append({
            "question_id": question_id + 1,
            "question": query,
            "expected_focus_area": expected_focus,
            "outcome": row["outcome"],
            "retrieval_method": "FAISS",
            "rank": result["rank"],
            "retrieved_focus_area": result["focus_area"],
            "source": result["source"],
            "passage": result["passage"],
            "similarity_score": result["similarity"],
            "reranker_score": None,
            "relevance_grade": None,
            "error_category": None,
            "reviewer_comments": None
        })

    # Record reranked results
    for _, result in reranked_results.iterrows():

        review_records.append({
            "question_id": question_id + 1,
            "question": query,
            "expected_focus_area": expected_focus,
            "outcome": row["outcome"],
            "retrieval_method": "Cross-Encoder",
            "rank": result["reranked_rank"],
            "retrieved_focus_area": result["focus_area"],
            "source": result["source"],
            "passage": result["passage"],
            "similarity_score": result["faiss_score"],
            "reranker_score": result["reranker_score"],
            "relevance_grade": None,
            "error_category": None,
            "reviewer_comments": None
        })

manual_review_df = pd.DataFrame(review_records)

print("Total review rows:", len(manual_review_df))
print("\nRetrieval methods:")
print(manual_review_df["retrieval_method"].value_counts())

display(
    manual_review_df[
        [
            "question_id",
            "retrieval_method",
            "rank",
            "retrieved_focus_area",
            "source"
        ]
    ].head(10)
)

Total review rows: 100

Retrieval methods:
retrieval_method
FAISS            50
Cross-Encoder    50
Name: count, dtype: int64


,question_id,retrieval_method,rank,retrieved_focus_area,source
0,1,FAISS,1,Klver-Bucy Syndrome,NINDS
1,1,FAISS,2,Klver-Bucy Syndrome,NINDS
2,1,FAISS,3,Klver-Bucy Syndrome,NINDS
3,1,FAISS,4,Kluver Bucy syndrome,GARD
4,1,FAISS,5,Klippel-Trenaunay Syndrome (KTS),NINDS
5,1,Cross-Encoder,1,Klippel-Trenaunay Syndrome (KTS),NINDS
6,1,Cross-Encoder,2,Krabbe Disease,NINDS
7,1,Cross-Encoder,3,Guillain-Barr Syndrome,NINDS
8,1,Cross-Encoder,4,Brown-Sequard Syndrome,NINDS
9,1,Cross-Encoder,5,Rett Syndrome,NINDS


In [18]:
review_path = (
    output_dir / "mq018_manual_evidence_review.csv"
)

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Review dataset saved:", review_path)
print("Total rows:", len(manual_review_df))

Review dataset saved: /home/sagemaker-user/medical_chatbot/data/processed/retrieval_evaluation/mq018_manual_evidence_review.csv
Total rows: 100


In [19]:
# MQ-018: Inspect full medical evidence for Question 1

pd.set_option("display.max_colwidth", None)

question_1 = manual_review_df[
    manual_review_df["question_id"] == 1
].copy()

print("VALIDATION QUESTION:")
print(question_1["question"].iloc[0])

print("\nEXPECTED MEDICAL CONDITION:")
print(question_1["expected_focus_area"].iloc[0])

for method in ["FAISS", "Cross-Encoder"]:

    print("\n" + "=" * 80)
    print("RETRIEVAL METHOD:", method)
    print("=" * 80)

    method_results = question_1[
        question_1["retrieval_method"] == method
    ].sort_values("rank")

    for _, row in method_results.iterrows():

        print(f"\nRANK: {row['rank']}")
        print("FOCUS AREA:", row["retrieved_focus_area"])
        print("SOURCE:", row["source"])
        print("PASSAGE:")
        print(row["passage"])
        print("-" * 80)

VALIDATION QUESTION:
what research (or clinical trials) is being done for Klver-Bucy Syndrome ?

EXPECTED MEDICAL CONDITION:
Klver-Bucy Syndrome

RETRIEVAL METHOD: FAISS

RANK: 1
FOCUS AREA: Klver-Bucy Syndrome
SOURCE: NINDS
PASSAGE:
Treatment is symptomatic and supportive, and may include drug therapy.
--------------------------------------------------------------------------------

RANK: 2
FOCUS AREA: Klver-Bucy Syndrome
SOURCE: NINDS
PASSAGE:
There is no cure for Klver-Bucy syndrome. The disorder is not life-threatening, but the patient can be difficult to manage. With treatment, symptoms may slowly decline.
--------------------------------------------------------------------------------

RANK: 3
FOCUS AREA: Klver-Bucy Syndrome
SOURCE: NINDS
PASSAGE:
Klver-Bucy syndrome is a rare behavioral impairment that is associated with damage to both of the anterior temporal lobes of the brain. It causes individuals to put objects in their mouths and engage in inappropriate sexual behavior. Ot

In [20]:
# MQ-018: Inspect passages individually

question_1 = manual_review_df[
    manual_review_df["question_id"] == 1
].sort_values(["retrieval_method", "rank"])

for _, row in question_1.iterrows():
    print(
        f"\n{row['retrieval_method']} | "
        f"Rank {row['rank']} | "
        f"{row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)


Cross-Encoder | Rank 1 | Klippel-Trenaunay Syndrome (KTS)
The NINDS supports research on congenital disorders such as KTS with the goal of finding new means to treat and prevent them.

Cross-Encoder | Rank 2 | Krabbe Disease
if it dramatically increases life expectancy in a mouse model of the disease. Also in a mouse mode, NINDS-funded scientists are testing a combined treatment approach that uses a harmless virus to increase protein production, along with blood stem cell transplantation and small-molecule-based drugs, to reduce neuroinflammation, cell death, and nerve cell degeneration seen in Krabbe disease.

Cross-Encoder | Rank 3 | Guillain-Barr Syndrome
Scientists are concentrating on finding new treatments and refining existing ones. Scientists are also looking at the workings of the immune system to find which cells are responsible for beginning and carrying out the attack on the nervous system. The fact that so many cases of Guillain-Barr begin after a viral or bacterial infec

In [21]:
remaining_faiss = question_1[
    (question_1["retrieval_method"] == "FAISS")
    & (question_1["rank"].between(3, 5))
]

for _, row in remaining_faiss.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])


FAISS Rank 3
Focus area: Klver-Bucy Syndrome
Passage: Klver-Bucy syndrome is a rare behavioral impairment that is associated with damage to both of the anterior temporal lobes of the brain. It causes individuals to put objects in their mouths and engage in inappropriate sexual behavior. Other symptoms may include visual agnosia (inability to visually recognize objects), loss of normal fear and anger responses, memory loss, distractibility, seizures, and dementia. The disorder may be associated with herpes encephalitis and trauma, which can result in brain damage.

FAISS Rank 4
Focus area: Kluver Bucy syndrome
Passage: Kluver Bucy syndrome is a rare behavioral impairment characterized by inappropriate sexual behaviors and mouthing of objects. Other signs and symptoms, include a diminished ability to visually recognize objects, loss of normal fear and anger responses, memory loss, distractibility, seizures, and dementia. It is associated with damage to the anterior temporal lobes of the

In [22]:
# MQ-018: Record human-reviewed annotations for Question 1

question_id = 1

annotations = {
    "FAISS": {
        1: (1, "Incorrect question intent"),
        2: (1, "Incorrect question intent"),
        3: (1, "Incorrect question intent"),
        4: (1, "Incorrect question intent"),
        5: (0, "Wrong medical condition"),
    },
    "Cross-Encoder": {
        1: (0, "Wrong medical condition"),
        2: (0, "Wrong medical condition"),
        3: (0, "Wrong medical condition"),
        4: (0, "Wrong medical condition"),
        5: (0, "Wrong medical condition"),
    }
}

for method, ranks in annotations.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == question_id)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[
            mask, "relevance_grade"
        ] = grade

        manual_review_df.loc[
            mask, "error_category"
        ] = category

        manual_review_df.loc[
            mask, "reviewer_comments"
        ] = (
            "Reviewed against the question about "
            "Klüver-Bucy syndrome research."
        )

# Save annotations
manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 1 annotations saved.")

display(
    manual_review_df.loc[
        manual_review_df["question_id"] == 1,
        [
            "retrieval_method",
            "rank",
            "relevance_grade",
            "error_category"
        ]
    ]
)

Question 1 annotations saved.


,retrieval_method,rank,relevance_grade,error_category
0,FAISS,1,1,Incorrect question intent
1,FAISS,2,1,Incorrect question intent
2,FAISS,3,1,Incorrect question intent
3,FAISS,4,1,Incorrect question intent
4,FAISS,5,0,Wrong medical condition
5,Cross-Encoder,1,0,Wrong medical condition
6,Cross-Encoder,2,0,Wrong medical condition
7,Cross-Encoder,3,0,Wrong medical condition
8,Cross-Encoder,4,0,Wrong medical condition
9,Cross-Encoder,5,0,Wrong medical condition


### MQ-018: Case Study 1 — Disease Identity vs Question Intent

**Question:** What research or clinical trials are being
conducted for Klüver-Bucy syndrome?

**FAISS findings:**
- Four passages discuss the correct medical condition.
- None directly addresses ongoing research or clinical trials.
- Mean relevance grade: 0.8/3.
- Precision@5 (grade >= 2): 0%.

**Cross-encoder findings:**
- All five passages discuss different medical conditions.
- Passages frequently match the research-related intent.
- Mean relevance grade: 0.0/3.
- Precision@5 (grade >= 2): 0%.

**Research interpretation:**
The cross-encoder prioritizes passages with similar
research-related language but fails to preserve the
specific medical condition in this example.

**Implication:**
Medical evidence retrieval should account for both
disease identity and question-specific intent.

**Limitation:**
This finding concerns one validation question and
does not establish a general failure mechanism.

In [23]:
question_2 = manual_review_df[
    manual_review_df["question_id"] == 2
].sort_values(["retrieval_method", "rank"])

print("QUESTION:", question_2["question"].iloc[0])

for _, row in question_2.iterrows():
    print(
        f"\n{row['retrieval_method']} | Rank {row['rank']} "
        f"| {row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)

QUESTION: What are the treatments for 3MC syndrome ?

Cross-Encoder | Rank 1 | 3-M syndrome
These resources address the diagnosis or management of 3-M syndrome: - Gene Review: Gene Review: 3-M Syndrome - Genetic Testing Registry: Three M syndrome 1 These resources from MedlinePlus offer information about the diagnosis and management of various health conditions: - Diagnostic Tests - Drug Therapy - Surgery and Rehabilitation - Genetic Counseling - Palliative Care

Cross-Encoder | Rank 2 | Chromosome 3p- syndrome
Special education services are often necessary for children with intellectual disability. Surgery may be required to treat certain physical abnormalities such as cleft palate or congenital heart defects, if present. Please speak to your healthcare provider if you have any questions about your personal medical management plan.

Cross-Encoder | Rank 3 | Chromosome 3p- syndrome
How might chromosome 3p- syndrome be treated? Because chromosome 3p- syndrome affects many different syst

In [24]:
# MQ-018: Question 2 — verified manual annotations

annotations_q2 = {
    "FAISS": {
        1: (1, "Incorrect question intent"),
        5: (0, "Wrong medical condition"),
    },
    "Cross-Encoder": {
        1: (0, "Disease identity confusion"),
        2: (0, "Wrong medical condition"),
        3: (0, "Wrong medical condition"),
        4: (0, "Wrong medical condition"),
        5: (0, "Wrong medical condition"),
    }
}

for method, ranks in annotations_q2.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == 2)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[mask, "relevance_grade"] = grade
        manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 2 verified annotations saved.")

Question 2 verified annotations saved.


In [25]:
remaining = manual_review_df[
    (manual_review_df["question_id"] == 2)
    & (manual_review_df["retrieval_method"] == "FAISS")
    & (manual_review_df["rank"].isin([2, 3, 4]))
].sort_values("rank")

for _, row in remaining.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])
    print("=" * 70)


FAISS Rank 2
Focus area: 3-M syndrome
Passage: These resources address the diagnosis or management of 3-M syndrome: - Gene Review: Gene Review: 3-M Syndrome - Genetic Testing Registry: Three M syndrome 1 These resources from MedlinePlus offer information about the diagnosis and management of various health conditions: - Diagnostic Tests - Drug Therapy - Surgery and Rehabilitation - Genetic Counseling - Palliative Care

FAISS Rank 3
Focus area: Chromosome 3p- syndrome
Passage: How might chromosome 3p- syndrome be treated? Because chromosome 3p- syndrome affects many different systems of the body, medical management is often provided by a team of doctors and other healthcare professionals. Treatment for this deletion varies based on the signs and symptoms present in each person. For example, children with delayed motor milestones (i.e. walking) and/or muscle problems may be referred for physical or occupational therapy. Severe feeding difficulties may be treated temporarily with a nasog

In [26]:
# MQ-018: Complete Question 2 manual annotations

remaining_annotations = {
    2: (0, "Disease identity confusion"),
    3: (0, "Wrong medical condition"),
    4: (1, "Incorrect question intent")
}

for rank, (grade, category) in remaining_annotations.items():

    mask = (
        (manual_review_df["question_id"] == 2)
        & (manual_review_df["retrieval_method"] == "FAISS")
        & (manual_review_df["rank"] == rank)
    )

    manual_review_df.loc[mask, "relevance_grade"] = grade
    manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 2 annotations completed and saved.")

display(
    manual_review_df.loc[
        manual_review_df["question_id"] == 2,
        [
            "retrieval_method",
            "rank",
            "relevance_grade",
            "error_category"
        ]
    ]
)

Question 2 annotations completed and saved.


,retrieval_method,rank,relevance_grade,error_category
10,FAISS,1,1,Incorrect question intent
11,FAISS,2,0,Disease identity confusion
12,FAISS,3,0,Wrong medical condition
13,FAISS,4,1,Incorrect question intent
14,FAISS,5,0,Wrong medical condition
15,Cross-Encoder,1,0,Disease identity confusion
16,Cross-Encoder,2,0,Wrong medical condition
17,Cross-Encoder,3,0,Wrong medical condition
18,Cross-Encoder,4,0,Wrong medical condition
19,Cross-Encoder,5,0,Wrong medical condition


In [27]:
question_3 = manual_review_df[
    manual_review_df["question_id"] == 3
].sort_values(["retrieval_method", "rank"])

print("QUESTION:", question_3["question"].iloc[0])

for _, row in question_3.iterrows():
    print(
        f"\n{row['retrieval_method']} | Rank {row['rank']} "
        f"| {row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)

QUESTION: what research (or clinical trials) is being done for Isaacs' Syndrome ?

Cross-Encoder | Rank 1 | Zellweger Syndrome
The National Institute of Neurological Disorders and Stroke (NINDS), and other institutes of the National Institutes of Health (NIH), conduct research exploring the molecular and genetic basis of Zellweger syndrome and the other PBDs, and also support additional research through grants to major research institutions across the country. Much of this research focuses on finding better ways to prevent, treat, and ultimately cure disorders such as Zellweger syndrome.

Cross-Encoder | Rank 2 | Guillain-Barr Syndrome
Scientists are concentrating on finding new treatments and refining existing ones. Scientists are also looking at the workings of the immune system to find which cells are responsible for beginning and carrying out the attack on the nervous system. The fact that so many cases of Guillain-Barr begin after a viral or bacterial infection suggests that certa

In [28]:
# MQ-018: Question 3 — verified annotations

annotations_q3 = {
    "FAISS": {
        1: (1, "Incorrect question intent"),
        5: (0, "Wrong medical condition")
    },
    "Cross-Encoder": {
        1: (0, "Wrong medical condition"),
        2: (0, "Wrong medical condition"),
        3: (0, "Wrong medical condition"),
        4: (0, "Wrong medical condition"),
        5: (0, "Wrong medical condition")
    }
}

for method, ranks in annotations_q3.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == 3)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[mask, "relevance_grade"] = grade
        manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 3 verified annotations saved.")

Question 3 verified annotations saved.


In [29]:
remaining_q3 = manual_review_df[
    (manual_review_df["question_id"] == 3)
    & (manual_review_df["retrieval_method"] == "FAISS")
    & (manual_review_df["rank"].isin([2, 3, 4]))
].sort_values("rank")

for _, row in remaining_q3.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])
    print("=" * 70)


FAISS Rank 2
Focus area: Isaacs' Syndrome
Passage: limited to cranial muscles. Speech and breathing may be affected if pharyngeal or laryngeal muscles are involved. Onset is between ages 15 and 60, with most individuals experiencing symptoms before age 40. There are hereditary and acquired (occurring from unknown causes) forms of the disorder. The acquired form occasionally develops in association with peripheral neuropathies or after radiation treatment, but more often is caused by an autoimmune condition. Autoimmune-mediated Issacs' syndrome is typically caused by antibodies that bind to potassium channels on the motor nerve. Issacs' syndrome is only one of several neurological conditions that can be caused by potassium channel antibodies.

FAISS Rank 3
Focus area: Isaacs' Syndrome
Passage: There is no cure for Isaacs' syndrome. The long-term prognosis for individuals with the disorder is uncertain.

FAISS Rank 4
Focus area: Angelman Syndrome
Passage: The NINDS supports and conducts

In [30]:
# MQ-018: Complete Question 3 annotations

remaining_q3_annotations = {
    2: (1, "Incorrect question intent"),
    3: (1, "Incorrect question intent"),
    4: (0, "Wrong medical condition")
}

for rank, (grade, category) in remaining_q3_annotations.items():

    mask = (
        (manual_review_df["question_id"] == 3)
        & (manual_review_df["retrieval_method"] == "FAISS")
        & (manual_review_df["rank"] == rank)
    )

    manual_review_df.loc[mask, "relevance_grade"] = grade
    manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 3 annotations completed and saved.")

Question 3 annotations completed and saved.


In [31]:
question_4 = manual_review_df[
    manual_review_df["question_id"] == 4
].sort_values(["retrieval_method", "rank"])

print("QUESTION:", question_4["question"].iloc[0])

for _, row in question_4.iterrows():
    print(
        f"\n{row['retrieval_method']} | Rank {row['rank']} "
        f"| {row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)

QUESTION: what research (or clinical trials) is being done for Lung Cancer ?

Cross-Encoder | Rank 1 | Non-Small Cell Lung Cancer
therapy. The combination of chemotherapy and radiation therapy given with a radiosensitizer is being studied in the treatment of non-small cell lung cancer. New combinations New combinations of treatments are being studied in clinical trials. Patients may want to think about taking part in a clinical trial. For some patients, taking part in a clinical trial may be the best treatment choice. Clinical trials are part of the cancer research process. Clinical trials are done to find out if new cancer treatments are safe and effective or better than the standard treatment. Many of today's standard treatments for cancer are based on earlier clinical trials. Patients who take part in a clinical trial may receive the standard treatment or be

Cross-Encoder | Rank 2 | Small Cell Lung Cancer
the standard treatment or be among the first to receive a new treatment. Pati

In [32]:
# MQ-018: Question 4 — provisional manual annotations

annotations_q4 = {
    "FAISS": {
        1: (2, "Relevant subtype evidence"),
        5: (2, "Relevant subtype evidence")
    },
    "Cross-Encoder": {
        1: (3, "Relevant subtype evidence"),
        2: (2, "Relevant subtype evidence"),
        3: (2, "Relevant subtype evidence"),
        4: (2, "Relevant subtype evidence"),
        5: (3, "Relevant subtype evidence")
    }
}

for method, ranks in annotations_q4.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == 4)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[mask, "relevance_grade"] = grade
        manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 4 verified passages annotated.")

Question 4 verified passages annotated.


In [33]:
remaining_q4 = manual_review_df[
    (manual_review_df["question_id"] == 4)
    & (manual_review_df["retrieval_method"] == "FAISS")
    & (manual_review_df["rank"].isin([2, 3, 4]))
].sort_values("rank")

for _, row in remaining_q4.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])
    print("=" * 70)


FAISS Rank 2
Focus area: Non-Small Cell Lung Cancer
Passage: based on earlier clinical trials. Patients who take part in a clinical trial may receive the standard treatment or be among the first to receive a new treatment. Patients who take part in clinical trials also help improve the way cancer will be treated in the future. Even when clinical trials do not lead to effective new treatments, they often answer important questions and help move research forward. Patients can enter clinical trials before, during, or after starting their cancer treatment. Some clinical trials only include patients who have not yet received treatment. Other trials test treatments for patients whose cancer has not gotten better. There are also clinical trials that test new ways to stop cancer from recurring (coming

FAISS Rank 3
Focus area: Small Cell Lung Cancer
Passage: the standard treatment or be among the first to receive a new treatment. Patients who take part in clinical trials also help improve the

In [34]:
# MQ-018: Complete Question 4 annotations

remaining_q4_annotations = {
    2: (2, "Relevant subtype evidence"),
    3: (2, "Relevant subtype evidence"),
    4: (2, "Relevant clinical-trial evidence")
}

for rank, (grade, category) in remaining_q4_annotations.items():

    mask = (
        (manual_review_df["question_id"] == 4)
        & (manual_review_df["retrieval_method"] == "FAISS")
        & (manual_review_df["rank"] == rank)
    )

    manual_review_df.loc[mask, "relevance_grade"] = grade
    manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 4 annotations completed and saved.")

Question 4 annotations completed and saved.


In [35]:
question_5 = manual_review_df[
    manual_review_df["question_id"] == 5
].sort_values(["retrieval_method", "rank"])

print("QUESTION:", question_5["question"].iloc[0])

for _, row in question_5.iterrows():
    print(
        f"\n{row['retrieval_method']} | Rank {row['rank']} "
        f"| {row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)

QUESTION: What are the genetic changes related to Wiskott-Aldrich syndrome ?

Cross-Encoder | Rank 1 | Beckwith-Wiedemann syndrome
The genetic causes of Beckwith-Wiedemann syndrome are complex. The condition usually results from the abnormal regulation of genes in a particular region of chromosome 11. People normally inherit one copy of this chromosome from each parent. For most genes on chromosome 11, both copies of the gene are expressed, or "turned on," in cells. For some genes, however, only the copy inherited from a person's father (the paternally inherited copy) is expressed. For other genes, only the copy inherited from a person's mother (the maternally inherited copy) is expressed. These parent-specific differences in gene expression are caused by a phenomenon called genomic imprinting. Abnormalities involving genes on chromosome 11 that undergo genomic imprinting are responsible for most cases of Beckwith-Wiedemann

Cross-Encoder | Rank 2 | Beckwith-Wiedemann syndrome
called g

In [36]:
# MQ-018: Question 5 — verified annotations

annotations_q5 = {
    "FAISS": {
        1: (2, "Incomplete genetic evidence"),
        5: (1, "Incorrect question intent")
    },
    "Cross-Encoder": {
        1: (0, "Wrong medical condition"),
        2: (0, "Wrong medical condition"),
        3: (0, "Wrong medical condition"),
        4: (0, "Wrong medical condition"),
        5: (0, "Wrong medical condition")
    }
}

for method, ranks in annotations_q5.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == 5)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[mask, "relevance_grade"] = grade
        manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 5 verified annotations saved.")

Question 5 verified annotations saved.


In [37]:
remaining_q5 = manual_review_df[
    (manual_review_df["question_id"] == 5)
    & (manual_review_df["retrieval_method"] == "FAISS")
    & (manual_review_df["rank"].isin([2, 3, 4]))
].sort_values("rank")

for _, row in remaining_q5.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])
    print("=" * 70)


FAISS Rank 2
Focus area: Wiskott-Aldrich syndrome
Passage: Wiskott-Aldrich syndrome is characterized by abnormal immune system function (immune deficiency) and a reduced ability to form blood clots. This condition primarily affects males. Individuals with Wiskott-Aldrich syndrome have microthrombocytopenia, which is a decrease in the number and size of blood cells involved in clotting (platelets). This platelet abnormality, which is typically present from birth, can lead to easy bruising or episodes of prolonged bleeding following minor trauma. Wiskott-Aldrich syndrome causes many types of white blood cells, which are part of the immune system, to be abnormal or nonfunctional, leading to an increased risk of several immune and inflammatory disorders. Many people with this condition develop eczema, an inflammatory skin disorder characterized by abnormal patches of red, irritated skin. Affected

FAISS Rank 3
Focus area: Wiskott-Aldrich syndrome
Passage: Many people with this condition d

In [38]:
# MQ-018: Complete Question 5 manual annotations

remaining_q5_annotations = {
    2: (1, "Incorrect question intent"),
    3: (1, "Incorrect question intent"),
    4: (1, "Incorrect question intent")
}

for rank, (grade, category) in remaining_q5_annotations.items():

    mask = (
        (manual_review_df["question_id"] == 5)
        & (manual_review_df["retrieval_method"] == "FAISS")
        & (manual_review_df["rank"] == rank)
    )

    manual_review_df.loc[mask, "relevance_grade"] = grade
    manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 5 annotations completed and saved.")

Question 5 annotations completed and saved.


In [39]:
question_6 = manual_review_df[
    manual_review_df["question_id"] == 6
].sort_values(["retrieval_method", "rank"])

print("QUESTION:", question_6["question"].iloc[0])

for _, row in question_6.iterrows():
    print(
        f"\n{row['retrieval_method']} | Rank {row['rank']} "
        f"| {row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)

QUESTION: Is Galactosialidosis inherited ?

Cross-Encoder | Rank 1 | Galactosialidosis
Galactosialidosis is an autosomal recessive lysosomal storage disorder caused by mutations in the CTSA gene. It is characterized by coarse facial features, macular cherry-red spots, angiokeratoma (dark red spots on the skin), vertebral deformities, epilepsy, action myoclonus, and ataxia. There are three different types of galactosialidosis: early infantile, late infantile and juvenile/adult. The three forms of galactosialidosis are distinguished by the age at which symptoms develop and the pattern of features.

Cross-Encoder | Rank 2 | Galactosialidosis
number of patients (when available) Abnormality of the macula 90% Coarse facial features 90% Cognitive impairment 90% Hearing impairment 90% Opacification of the corneal stroma 90% Seizures 90% Short stature 90% Skeletal dysplasia 90% Hepatosplenomegaly 7.5% Autosomal recessive inheritance - Cherry red spot of the macula - Conjunctival telangiectasia 

In [40]:
# MQ-018: Question 6 — verified annotations

annotations_q6 = {
    "FAISS": {
        1: (3, "Direct inheritance evidence"),
        5: (1, "Incorrect question intent")
    },
    "Cross-Encoder": {
        1: (3, "Direct inheritance evidence"),
        2: (3, "Direct inheritance evidence"),
        3: (2, "Indirect genetic evidence"),
        4: (2, "Indirect genetic evidence"),
        5: (2, "Indirect genetic evidence")
    }
}

for method, ranks in annotations_q6.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == 6)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[mask, "relevance_grade"] = grade
        manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 6 verified annotations saved.")

Question 6 verified annotations saved.


In [41]:
remaining_q6 = manual_review_df[
    (manual_review_df["question_id"] == 6)
    & (manual_review_df["retrieval_method"] == "FAISS")
    & (manual_review_df["rank"].isin([2, 3, 4]))
].sort_values("rank")

for _, row in remaining_q6.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])
    print("=" * 70)


FAISS Rank 2
Focus area: galactosialidosis
Passage: symptoms begin to develop varies widely among affected individuals, but the average age is 16. This form is typically associated with a normal life expectancy.

FAISS Rank 3
Focus area: galactosialidosis
Passage: Galactosialidosis is a condition that affects many areas of the body. The three forms of galactosialidosis are distinguished by the age at which symptoms develop and the pattern of features. The early infantile form of galactosialidosis is associated with extensive swelling caused by fluid accumulation before birth (hydrops fetalis), a soft out-pouching in the lower abdomen (an inguinal hernia), and an enlarged liver and spleen (hepatosplenomegaly). Additional features of this form include abnormal bone development (dysostosis multiplex) and distinctive facial features that are often described as "coarse." Some infants have an enlarged heart (cardiomegaly); an eye abnormality called a cherry-red spot, which can be identified

In [42]:
# MQ-018: Complete Question 6 annotations

remaining_q6_annotations = {
    2: (1, "Incorrect question intent"),
    3: (1, "Incorrect question intent"),
    4: (1, "Incorrect question intent")
}

for rank, (grade, category) in remaining_q6_annotations.items():

    mask = (
        (manual_review_df["question_id"] == 6)
        & (manual_review_df["retrieval_method"] == "FAISS")
        & (manual_review_df["rank"] == rank)
    )

    manual_review_df.loc[mask, "relevance_grade"] = grade
    manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 6 annotations completed and saved.")

Question 6 annotations completed and saved.


In [43]:
question_7 = manual_review_df[
    manual_review_df["question_id"] == 7
].sort_values(["retrieval_method", "rank"])

print("QUESTION:", question_7["question"].iloc[0])

for _, row in question_7.iterrows():
    print(
        f"\n{row['retrieval_method']} | Rank {row['rank']} "
        f"| {row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)

QUESTION: What are the symptoms of Coronary Microvascular Disease ?

Cross-Encoder | Rank 1 | Coronary Microvascular Disease
Your doctor will diagnose coronary microvascular disease (MVD) based on your medical history, a physical exam, and test results. He or she will check to see whether you have any risk factors for heart disease. For example, your doctor may measure your weight and height to check for overweight or obesity. He or she also may recommend tests for high blood cholesterol, metabolic syndrome, and diabetes. Your doctor may ask you to describe any chest pain, including when it started and how it changed during physical activity or periods of stress. He or she also may ask about other symptoms, such as fatigue (tiredness), lack of energy, and shortness of breath. Women may be asked about their menopausal status. Specialists

Cross-Encoder | Rank 2 | Coronary Microvascular Disease
Coronary microvascular disease (MVD) is heart disease that affects the tiny coronary (heart) a

In [44]:
# MQ-018: Question 7 — verified annotations

annotations_q7 = {
    "FAISS": {
        1: (3, "Direct symptom evidence"),
        5: (1, "Incorrect question intent")
    },
    "Cross-Encoder": {
        1: (3, "Direct symptom evidence"),
        2: (1, "Incorrect question intent"),
        3: (1, "Incorrect question intent"),
        4: (2, "Partial symptom evidence"),
        5: (3, "Direct symptom evidence")
    }
}

for method, ranks in annotations_q7.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == 7)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[mask, "relevance_grade"] = grade
        manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 7 visible passages annotated.")

Question 7 visible passages annotated.


In [45]:
remaining_q7 = manual_review_df[
    (manual_review_df["question_id"] == 7)
    & (manual_review_df["retrieval_method"] == "FAISS")
    & (manual_review_df["rank"].isin([2, 3, 4]))
].sort_values("rank")

for _, row in remaining_q7.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])
    print("=" * 70)


FAISS Rank 2
Focus area: Coronary Microvascular Disease
Passage: flow through a coronary artery. In coronary MVD, however, the heart's tiny arteries are affected. Plaque doesn't create blockages in these vessels as it does in the heart's large arteries. Coronary Microvascular Disease Overview Both men and women who have coronary microvascular disease often have diabetes or high blood pressure. Some people who have coronary microvascular disease may have inherited heart muscle diseases. Diagnosing coronary microvascular disease has been a challenge for doctors. Standard tests used to diagnose coronary heart disease arent designed to detect coronary microvascular disease. More research is needed to find the best diagnostic tests and treatments for thedisease. Outlook Most of what is known about coronary MVD comes from the National Heart, Lung, and Blood Institute's

FAISS Rank 3
Focus area: Coronary Microvascular Disease
Passage: such as fatigue (tiredness), lack of energy, and shortnes

In [46]:
# MQ-018: Complete Question 7 annotations

remaining_q7_annotations = {
    2: (1, "Incorrect question intent"),
    3: (2, "Partial symptom evidence"),
    4: (3, "Direct symptom evidence")
}

for rank, (grade, category) in remaining_q7_annotations.items():

    mask = (
        (manual_review_df["question_id"] == 7)
        & (manual_review_df["retrieval_method"] == "FAISS")
        & (manual_review_df["rank"] == rank)
    )

    manual_review_df.loc[mask, "relevance_grade"] = grade
    manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 7 annotations completed and saved.")

Question 7 annotations completed and saved.


In [47]:
question_8 = manual_review_df[
    manual_review_df["question_id"] == 8
].sort_values(["retrieval_method", "rank"])

print("QUESTION:", question_8["question"].iloc[0])

for _, row in question_8.iterrows():
    print(
        f"\n{row['retrieval_method']} | Rank {row['rank']} "
        f"| {row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)

QUESTION: What is (are) Schwartz-Jampel syndrome ?

Cross-Encoder | Rank 1 | Schwartz Jampel syndrome type 1
Schwartz Jampel syndrome type 1 (SJS1) is a genetic disorder that affects bone and muscle development. Signs and symptoms may include muscle weakness and stiffness, abnormal bone development, joint contractures, short stature, small, fixed facial features, and eye abnormalities (some of which may impair vision). SJS1 can be divided into two subtypes differentiated by severity and age of onset. Type 1A, considered classic SJS, is the most commonly recognized type. Individuals with type 1A typically develop more mild symptoms later in childhood, while individuals with type 1B have symptoms that are more severe and are apparent immediately after birth. SJS1 is caused by mutations in the HSPG2 gene which makes a protein called perlecan. SJS1 is thought to be inherited

Cross-Encoder | Rank 2 | Schwartz Jampel syndrome type 1
is caused by mutations in the HSPG2 gene which makes a pro

In [48]:
# MQ-018: Question 8 — visible passage annotations

annotations_q8 = {
    "FAISS": {
        1: (2, "Partial disease definition"),
        5: (1, "Incomplete disease description")
    },
    "Cross-Encoder": {
        1: (3, "Direct disease definition"),
        2: (2, "Partial disease definition"),
        3: (2, "Partial disease definition"),
        4: (1, "Historical disease-name confusion"),
        5: (3, "Direct disease definition")
    }
}

for method, ranks in annotations_q8.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == 8)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[mask, "relevance_grade"] = grade
        manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 8 visible passages annotated.")

Question 8 visible passages annotated.


In [49]:
remaining_q8 = manual_review_df[
    (manual_review_df["question_id"] == 8)
    & (manual_review_df["retrieval_method"] == "FAISS")
    & (manual_review_df["rank"].isin([2, 3, 4]))
].sort_values("rank")

for _, row in remaining_q8.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])
    print("=" * 70)


FAISS Rank 2
Focus area: Schwartz-Jampel syndrome
Passage: in the amount or function of this protein disrupts the normal development of cartilage and bone tissue, which underlies chondrodysplasia in affected individuals. A reduced amount of functional perlecan at the neuromuscular junction likely alters the balance of other molecules that signal when muscles should contract and when they should relax. As a result, muscle contraction is triggered continuously, leading to myotonia.

FAISS Rank 3
Focus area: Schwartz Jampel syndrome type 1
Passage: 7. 5 % myalgia 7. 5 % nephrolithiasis 7. 5 % neurological speech impairment 7. 5 % odontogenic neoplasm 7. 5 % pectus excavatum 7. 5 % polyhydramnios 7. 5 % prenatal movement abnormality 7. 5 % protrusio acetabuli 7. 5 % pulmonary hypertension 7. 5 % respiratory insufficiency 7. 5 % skeletal muscle atrophy 7. 5 % sprengel anomaly 7. 5 % testicular torsion 7. 5 %

FAISS Rank 4
Focus area: Schwartz Jampel syndrome type 1
Passage: having a change

In [50]:
# MQ-018: Complete Question 8 annotations

remaining_q8_annotations = {
    2: (2, "Partial disease definition"),
    3: (1, "Incomplete disease description"),
    4: (1, "Incomplete disease description")
}

for rank, (grade, category) in remaining_q8_annotations.items():

    mask = (
        (manual_review_df["question_id"] == 8)
        & (manual_review_df["retrieval_method"] == "FAISS")
        & (manual_review_df["rank"] == rank)
    )

    manual_review_df.loc[mask, "relevance_grade"] = grade
    manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 8 annotations completed and saved.")

Question 8 annotations completed and saved.


In [51]:
question_9 = manual_review_df[
    manual_review_df["question_id"] == 9
].sort_values(["retrieval_method", "rank"])

print("QUESTION:", question_9["question"].iloc[0])

for _, row in question_9.iterrows():
    print(
        f"\n{row['retrieval_method']} | Rank {row['rank']} "
        f"| {row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)

QUESTION: What is (are) Menetrier disease ?

Cross-Encoder | Rank 1 | Menetrier disease
What are the signs and symptoms of Menetrier disease? Although some patients with Mntrier disease may not experience symptoms, most patients have stomach pain, diarrhea, weight loss, peripheral edema, and sometimes bleeding. The Human Phenotype Ontology provides the following list of signs and symptoms for Menetrier disease. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Abdominal pain 90% Edema of the lower limbs 90% Nausea and vomiting 90% Anorexia 50% Gastrointestinal hemorrhage 50% Hypoproteinemia 50% Iron deficiency anemia 50% Weight loss 50% Sepsis

Cross-Encoder | Rank 2 | Menetrier disease
What causes Mntrier disease? The exact cause of Mntrier disease is un

In [52]:
# MQ-018: Question 9 — visible passage annotations

annotations_q9 = {
    "FAISS": {
        1: (1, "Incorrect question intent"),
        5: (2, "Partial disease definition")
    },
    "Cross-Encoder": {
        1: (2, "Partial disease definition"),
        2: (2, "Partial disease definition"),
        3: (1, "Incorrect question intent"),
        4: (2, "Partial disease definition"),
        5: (1, "Non-clinical resource instructions")
    }
}

for method, ranks in annotations_q9.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == 9)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[mask, "relevance_grade"] = grade
        manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 9 visible passages annotated.")

Question 9 visible passages annotated.


In [53]:
remaining_q9 = manual_review_df[
    (manual_review_df["question_id"] == 9)
    & (manual_review_df["retrieval_method"] == "FAISS")
    & (manual_review_df["rank"].isin([2, 3, 4]))
].sort_values("rank")

for _, row in remaining_q9.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])
    print("=" * 70)


FAISS Rank 2
Focus area: Menkes Disease
Passage: easily. There is often extensive neurodegeneration in the gray matter of the brain. Arteries in the brain may be twisted with frayed and split inner walls. This can lead to rupture or blockage of the arteries. Weakened bones (osteoporosis) may result in fractures.

FAISS Rank 3
Focus area: Menetrier disease
Passage: What causes Mntrier disease? The exact cause of Mntrier disease is unknown. However, it has been associated with cytomegalovirus (CMV) infection in children and Heliobacter pylori (H. pylori) infection in adults. In addition, some have suggested that overexpression of a type of growth factor called the transforming growth factor-, which is found in a specific part of the stomach, the superficial gastric epithelium, might play a role.

FAISS Rank 4
Focus area: Menetrier disease
Passage: a complete document, while information on other studies is available as a summary abstract. To obtain the full article, contact a medical/uni

In [54]:
# MQ-018: Complete Question 9 annotations

remaining_q9_annotations = {
    2: (0, "Wrong medical condition"),
    3: (2, "Partial disease definition"),
    4: (1, "Non-clinical resource instructions")
}

for rank, (grade, category) in remaining_q9_annotations.items():

    mask = (
        (manual_review_df["question_id"] == 9)
        & (manual_review_df["retrieval_method"] == "FAISS")
        & (manual_review_df["rank"] == rank)
    )

    manual_review_df.loc[mask, "relevance_grade"] = grade
    manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 9 annotations completed and saved.")

Question 9 annotations completed and saved.


In [55]:
question_10 = manual_review_df[
    manual_review_df["question_id"] == 10
].sort_values(["retrieval_method", "rank"])

print("QUESTION:", question_10["question"].iloc[0])

for _, row in question_10.iterrows():
    print(
        f"\n{row['retrieval_method']} | Rank {row['rank']} "
        f"| {row['retrieved_focus_area']}"
    )
    print(row["passage"])
    print("=" * 60)

QUESTION: What causes Hanhart syndrome ?

Cross-Encoder | Rank 1 | Hanhart syndrome
Hanhart syndrome is a rare condition that primarily affects the craniofacial region and the limbs (arms and legs). People affected by this condition are often born with a short, incompletely developed tongue; absent or partially missing fingers and/or toes; abnormalities of the arms and/or legs; and an extremely small jaw. The severity of these physical abnormalities varies greatly among affected people, and children with this condition often have some, but not all, of the symptoms. The cause of Hanhart syndrome is not fully understood. Treatment depends on the signs and symptoms present in each person.

Cross-Encoder | Rank 2 | Hanhart syndrome
What are the signs and symptoms of Hanhart syndrome? The signs and symptoms of Hanhart syndrome vary, but may include: Small mouth Short, incompletely developed tongue (hypoglossia) Absent, partially missing, or shortened fingers and/or toes Jaw abnormalities su

In [56]:
# MQ-018: Question 10 — visible passage annotations

annotations_q10 = {
    "FAISS": {
        1: (3, "Direct cause evidence"),
        5: (1, "Incorrect question intent")
    },
    "Cross-Encoder": {
        1: (3, "Direct cause evidence"),
        2: (1, "Incorrect question intent"),
        3: (1, "Incorrect question intent"),
        4: (1, "Incorrect question intent"),
        5: (0, "Wrong medical condition")
    }
}

for method, ranks in annotations_q10.items():
    for rank, (grade, category) in ranks.items():

        mask = (
            (manual_review_df["question_id"] == 10)
            & (manual_review_df["retrieval_method"] == method)
            & (manual_review_df["rank"] == rank)
        )

        manual_review_df.loc[mask, "relevance_grade"] = grade
        manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 10 visible passages annotated.")

Question 10 visible passages annotated.


In [57]:
remaining_q10 = manual_review_df[
    (manual_review_df["question_id"] == 10)
    & (manual_review_df["retrieval_method"] == "FAISS")
    & (manual_review_df["rank"].isin([2, 3, 4]))
].sort_values("rank")

for _, row in remaining_q10.iterrows():
    print(f"\nFAISS Rank {row['rank']}")
    print("Focus area:", row["retrieved_focus_area"])
    print("Passage:", row["passage"])
    print("=" * 70)


FAISS Rank 2
Focus area: Hanhart syndrome
Passage: due to the craniofacial abnormalities listed above. The severity of the physical abnormalities associated with Hanhart syndrome varies greatly among affected people, and children with this disorder often have some, but not all, of the symptoms. The Human Phenotype Ontology provides the following list of signs and symptoms for Hanhart syndrome. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Narrow mouth 90% Upper limb phocomelia 90% Abnormality of the fingernails 50% Brachydactyly syndrome 50% Cleft palate 50% Finger syndactyly 50% Reduced number of

FAISS Rank 3
Focus area: Hanhart syndrome
Passage: the fractions may be different if another group of patients are examined. Sometimes, no information on 

In [58]:
# MQ-018: Complete Question 10 annotations

remaining_q10_annotations = {
    2: (1, "Incorrect question intent"),
    3: (1, "Non-clinical frequency explanation"),
    4: (1, "Incorrect question intent")
}

for rank, (grade, category) in remaining_q10_annotations.items():

    mask = (
        (manual_review_df["question_id"] == 10)
        & (manual_review_df["retrieval_method"] == "FAISS")
        & (manual_review_df["rank"] == rank)
    )

    manual_review_df.loc[mask, "relevance_grade"] = grade
    manual_review_df.loc[mask, "error_category"] = category

manual_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Question 10 annotations completed and saved.")

Question 10 annotations completed and saved.


In [59]:
import pandas as pd

reviewed_df = pd.read_csv(review_path)

reviewed_df["relevance_grade"] = pd.to_numeric(
    reviewed_df["relevance_grade"],
    errors="coerce"
)

assert len(reviewed_df) == 100, "Expected 100 review rows."
assert reviewed_df["relevance_grade"].notna().all(), (
    "Some relevance grades are missing."
)
assert reviewed_df["relevance_grade"].between(0, 3).all()

reviewed_df["relevant"] = reviewed_df["relevance_grade"] >= 2

summary = (
    reviewed_df.groupby("retrieval_method")
    .agg(
        passages=("relevant", "size"),
        relevant_passages=("relevant", "sum"),
        precision_at_5=("relevant", "mean"),
        mean_relevance_grade=("relevance_grade", "mean")
    )
)

summary["precision_at_5"] *= 100

print("MQ-018 FINAL MANUAL EVALUATION")
display(summary.round(3))

print("\nERROR CATEGORY DISTRIBUTION")
display(
    pd.crosstab(
        reviewed_df["error_category"],
        reviewed_df["retrieval_method"]
    )
)

MQ-018 FINAL MANUAL EVALUATION


,passages,relevant_passages,precision_at_5,mean_relevance_grade
retrieval_method,,,,
Cross-Encoder,50,21,42.0,1.18
FAISS,50,15,30.0,1.24



ERROR CATEGORY DISTRIBUTION


retrieval_method,Cross-Encoder,FAISS
error_category,,
Direct cause evidence,1,1
Direct disease definition,2,0
Direct inheritance evidence,2,1
Direct symptom evidence,2,2
Disease identity confusion,1,1
Historical disease-name confusion,1,0
Incomplete disease description,0,3
Incomplete genetic evidence,0,1
Incorrect question intent,6,23


In [60]:
summary_path = output_dir / "mq018_manual_evaluation_summary.csv"
errors_path = output_dir / "mq018_error_distribution.csv"

summary.to_csv(summary_path)
pd.crosstab(
    reviewed_df["error_category"],
    reviewed_df["retrieval_method"]
).to_csv(errors_path)

print("MQ-018 completed.")
print("Summary:", summary_path)
print("Errors:", errors_path)

MQ-018 completed.
Summary: /home/sagemaker-user/medical_chatbot/data/processed/retrieval_evaluation/mq018_manual_evaluation_summary.csv
Errors: /home/sagemaker-user/medical_chatbot/data/processed/retrieval_evaluation/mq018_error_distribution.csv
